# 05 — Customer value of revealed-preference ranking

**Part 6.** If a platform ranks each participant's dates using **stated** (`*1_1`) vs **revealed** (Part 3 logit) weights, how much does the **yes rate on top picks** change?

- Same sample as [`02_core_model`](02_core_model.ipynb): 452 people, complete six ratings + stated prefs, both yes and no decisions.
- Scores = weighted sum of partner ratings; no leakage (`match`, `like`, `prob` excluded).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.customer_value import run_customer_value, save_outputs

analysis = run_customer_value(n_bootstrap=1000)
save_outputs(analysis)
print(f"People: {analysis['n_people']}, decisions: {analysis['n_decisions']}")

People: 452, decisions: 6222


In [2]:
analysis["summary"].round(4)

rule,gender_label,k,n_people,revealed,stated,lift_pp,lift_ratio
0,All,1,452,0.9027,0.8717,0.0310,1.0355
1,All,3,452,0.8027,0.7828,0.0199,1.0254
2,Men,1,229,0.8865,0.8734,0.0131,1.0150
3,Men,3,229,0.8275,0.8159,0.0116,1.0143
4,Women,1,223,0.9193,0.8700,0.0493,1.0567
5,Women,3,223,0.7773,0.7489,0.0284,1.0379


In [3]:
analysis["bootstrap"].round(4)

,k,mean_lift_pp,ci_low,ci_high,n_draws
0,1,0.0315,0.0088,0.0575,1000
1,3,0.0200,0.0052,0.0354,1000


In [ ]:
analysis["top1_agreement"].round(4)

## Lift where #1 changes (~30%)

Pooled lift mixes **~70%** of people with **zero** top-1 lift (same #1 partner). Below: top-k yes rates by segment (`customer_value_lift_by_segment.csv`). Figure: `customer_value_topk_different_top1.png`.

In [ ]:
cols = ["segment", "gender_label", "k", "n_people", "stated", "revealed", "lift_pp"]
analysis["summary_by_segment"][cols].round(4)

## Managerial read (CLV / engagement)

Re-ranking by **revealed** weights lifts yes rates on top-1 and top-3 dates versus **stated** weights (pooled: `customer_value_topk_yes_rate.png`; **where #1 differs:** `customer_value_topk_different_top1.png`). This is a **historical replay** on one's own dates—not a live A/B test—but it quantifies the engagement upside matching platforms might capture if survey weights under-weight appearance and over-weight sincerity/intelligence.

Bootstrap CIs on lift (clustered by person) are in `outputs/tables/customer_value_bootstrap.csv` (`segment=all` and `different_top1`).

### FAQ: high yes on both top-1; same person?

- Stated vs revealed **#1 is the same partner ~70%** of the time; **~30%** re-ranking changes who is first (`customer_value_top1_agreement.csv`).
- Both top-1 yes rates stay high (~87% / ~90%) because stated weights still pick someone usually rated well; many participants said yes to several dates.
- **Lift comes from the ~30%** where #1 differs: top-1 yes **~78% → ~88%** (**+10 pp**; bootstrap CI in `customer_value_bootstrap.csv`). Same #1 → **0 pp** top-1 lift. Jiayi's **+20 pp** gap is **importance (0–100)**, not yes rate.